# Inference Session Test Script

Tests the full session lifecycle: create → multi-turn conversation → close.
All parameters are configurable in the cell below.

In [7]:
# ============================================================
# CONFIGURATION — Change these values as needed
# ============================================================
from datetime import datetime

# BASE_URL = "http://localhost:3005"
# API_KEY = "sk_V62D7QxaxZmcn5CIgjyPHqssTDsFyetLbtstZDrF6RcGYAVq"  # Replace with a valid tenant API key

BASE_URL = "https://dhdaqqvh7yj46.cloudfront.net"
API_KEY = "sk_22D_vhQfdyK0mmHDsW97ukefLCDsbB6K0ipg-MjGs_cDidkQ"  # Replace with a valid tenant API key

SESSION_NAME = f"notebook-test-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
CHANNEL = "API"
CHANNEL_METADATA = {
    "endUserId": "test-user-1",
    "source": "jupyter-notebook",
    "testRun": True,
}

# Conversation turns — add/remove/modify as needed
CONVERSATION_TURNS = [
    "Hi, I forgot my password. Can you help me reset it?",
    "My email is user@example.com",
    "I don't see the reset email in my inbox. What should I do?",
    "Got it, that worked. Thanks!",
]

# Streaming mode (True for SSE stream, False for JSON response)
STREAM = False

# Delay between turns (seconds) — simulates real user pacing
TURN_DELAY = 2

print(f"Target: {BASE_URL}")
print(f"Session: {SESSION_NAME}")
print(f"Turns: {len(CONVERSATION_TURNS)}")

Target: https://dhdaqqvh7yj46.cloudfront.net
Session: notebook-test-20260523-115841
Turns: 4


In [8]:
import requests
import json
import time

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

def api_post(path: str, body: dict) -> requests.Response:
    url = f"{BASE_URL}{path}"
    resp = requests.post(url, headers=HEADERS, json=body)
    return resp

def api_get(path: str) -> requests.Response:
    url = f"{BASE_URL}{path}"
    resp = requests.get(url, headers=HEADERS)
    return resp

print("Helpers loaded.")

Helpers loaded.


## Step 1: Create Session

In [9]:
create_resp = api_post("/api/v1/inference/sessions", {
    "name": SESSION_NAME,
    "channel": CHANNEL,
    "channelMetadata": CHANNEL_METADATA,
})

assert create_resp.status_code == 200 or create_resp.status_code == 201, (
    f"Session creation failed: {create_resp.status_code} — {create_resp.text}"
)

session_data = create_resp.json()
SESSION_ID = session_data["id"]

print(f"Session created: {SESSION_ID}")
print(f"Status: {session_data.get('status', 'unknown')}")
print(f"Channel: {session_data.get('channel', CHANNEL)}")
print(json.dumps(session_data, indent=2, default=str))

Session created: cmphywma60006lc01wscdlaew
Status: active
Channel: API
{
  "id": "cmphywma60006lc01wscdlaew",
  "apiKeyId": "cmphrce9v0009n301dlm5n12r",
  "tenantId": "cmpegedyc0003k301yh1frmj3",
  "agentId": "cmphr7nda0001n301wg6vkdkl",
  "agentVersionId": null,
  "name": "notebook-test-20260523-115841",
  "channel": "API",
  "channelMetadata": {
    "source": "jupyter-notebook",
    "testRun": true,
    "endUserId": "test-user-1"
  },
  "status": "active",
  "idleExpiresAt": "2026-05-23T06:58:42.317Z",
  "endedAt": null,
  "endReason": null,
  "createdAt": "2026-05-23T06:28:42.318Z",
  "updatedAt": "2026-05-23T06:28:42.318Z"
}


## Step 2: Multi-Turn Conversation

In [10]:
responses = []

for i, user_message in enumerate(CONVERSATION_TURNS, 1):
    print(f"\n{'='*60}")
    print(f"Turn {i}/{len(CONVERSATION_TURNS)}")
    print(f"{'='*60}")
    print(f"\n👤 User: {user_message}\n")

    start_time = time.time()

    turn_resp = api_post("/api/v1/inference", {
        "sessionId": SESSION_ID,
        "messages": [{"role": "user", "content": user_message}],
        "stream": STREAM,
    })

    elapsed = time.time() - start_time

    if turn_resp.status_code != 200:
        print(f"ERROR: {turn_resp.status_code} — {turn_resp.text}")
        responses.append({"turn": i, "error": turn_resp.text, "latency": elapsed})
        continue

    if STREAM:
        # For streaming, collect the full text from SSE events
        full_text = ""
        for line in turn_resp.iter_lines(decode_unicode=True):
            if line.startswith("data: "):
                data = line[6:]
                if data == "[DONE]":
                    break
                try:
                    event = json.loads(data)
                    if event.get("type") == "text-delta":
                        full_text += event.get("delta", "")
                except json.JSONDecodeError:
                    pass
        assistant_content = full_text
    else:
        result = turn_resp.json()
        assistant_content = result.get("content", result.get("text", str(result)))

    print(f"🤖 Assistant: {assistant_content[:300]}{'...' if len(assistant_content) > 300 else ''}")
    print(f"\n⏱️  Latency: {elapsed:.2f}s")

    responses.append({
        "turn": i,
        "user": user_message,
        "assistant": assistant_content,
        "latency": elapsed,
    })

    if i < len(CONVERSATION_TURNS):
        time.sleep(TURN_DELAY)

print(f"\n\n{'='*60}")
print(f"Conversation complete — {len(responses)} turns")
print(f"Total latency: {sum(r['latency'] for r in responses):.2f}s")
print(f"Avg latency: {sum(r['latency'] for r in responses) / len(responses):.2f}s")


Turn 1/4

👤 User: Hi, I forgot my password. Can you help me reset it?

🤖 Assistant: Hey there! No worries, I can help you get back into your account.

Could you let me know the email address (or username) you use for SMC? Once I have that, I’ll send a password‑reset link straight to you. 🚀

If you prefer, I can also walk you through the reset steps right here—just let me know what ...

⏱️  Latency: 1.79s

Turn 2/4

👤 User: My email is user@example.com

🤖 Assistant: Got it—thanks for confirming your email!

Here’s what you need to do next:

1. **Open the SMC login page** – [smc.com/login](https://smc.com/login)  
2. Click **“Forgot password?”** beneath the sign‑in fields.  
3. **Enter `user@example.com`** in the box and hit **Submit**.  
4. You’ll receive an em...

⏱️  Latency: 2.41s

Turn 3/4

👤 User: I don't see the reset email in my inbox. What should I do?

🤖 Assistant: I’m sorry the reset email didn’t show up – let’s get this sorted quickly.

### 1️⃣ Double‑check the usual spots
-

## Step 3: Verify Session State

In [11]:
session_resp = api_get(f"/api/v1/inference/sessions/{SESSION_ID}")

assert session_resp.status_code == 200, (
    f"Failed to fetch session: {session_resp.status_code} — {session_resp.text}"
)

session_state = session_resp.json()

print(f"Session ID: {session_state['id']}")
print(f"Status: {session_state.get('status', 'unknown')}")
print(f"Messages: {len(session_state.get('messages', []))}")
print(f"Idle expires: {session_state.get('idleExpiresAt', 'N/A')}")
print(f"\nMessage roles: {[m['role'] for m in session_state.get('messages', [])]}")

Session ID: cmphywma60006lc01wscdlaew
Status: active
Messages: 8
Idle expires: 2026-05-23T06:58:57.798Z

Message roles: ['user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant']


## Step 4: Close Session

In [12]:
close_resp = api_post(f"/api/v1/inference/sessions/{SESSION_ID}/close", {})

print(f"Close status: {close_resp.status_code}")

if close_resp.status_code == 204:
    print("Session closed successfully.")
    print("Analytics job has been enqueued — results will appear shortly.")
else:
    print(f"Response: {close_resp.text}")

Close status: 204
Session closed successfully.
Analytics job has been enqueued — results will appear shortly.


## Step 5: Verify Final State & Analytics

In [13]:
# Wait a moment for the analytics worker to process
ANALYTICS_WAIT = 10  # seconds to wait for analytics worker
print(f"Waiting {ANALYTICS_WAIT}s for analytics worker...")
time.sleep(ANALYTICS_WAIT)

# Use the dashboard API (not the inference API) since closed sessions return 410 on the inference endpoint
final_resp = api_get(f"/api/sessions/{SESSION_ID}")

if final_resp.status_code == 200:
    final = final_resp.json()
    session_info = final.get("session", final)
    messages = final.get("messages", [])
    analytics = final.get("analytics")

    print(f"\nFinal session state:")
    print(f"  Status: {session_info.get('status')}")
    print(f"  End reason: {session_info.get('endReason')}")
    print(f"  Ended at: {session_info.get('endedAt')}")
    print(f"  Messages: {len(messages)}")

    if analytics:
        print(f"\nAnalytics:")
        print(f"  Sentiment: {analytics.get('sentiment')}")
        print(f"  Resolved: {analytics.get('isResolved')}")
        print(f"  Confidence: {analytics.get('confidenceScore')}")
        print(f"  Language: {analytics.get('language')}")
        print(f"  Summary: {analytics.get('summary', 'N/A')[:200]}")
    else:
        print(f"\nAnalytics not yet available (worker may still be processing).")
        print(f"Re-run this cell after a few more seconds.")
elif final_resp.status_code in (401, 403):
    print("Dashboard API requires auth session (cookie-based).")
    print("Use the inference API with a longer wait, or check the dashboard UI directly.")
    print(f"\nDashboard URL: {BASE_URL}/sessions/{SESSION_ID}")
else:
    print(f"Session fetch returned: {final_resp.status_code}")
    print(final_resp.text)

Waiting 10s for analytics worker...


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## Summary

In [ ]:
print("\n" + "="*60)
print("SESSION TEST SUMMARY")
print("="*60)
print(f"\nSession ID:  {SESSION_ID}")
print(f"Endpoint:    {BASE_URL}")
print(f"Turns:       {len(CONVERSATION_TURNS)}")
print(f"Stream:      {STREAM}")
print(f"\nPer-turn latency:")
for r in responses:
    status = '✓' if 'assistant' in r else '✗'
    print(f"  Turn {r['turn']}: {r['latency']:.2f}s {status}")
print(f"\nDashboard URL: {BASE_URL}/sessions/{SESSION_ID}")
print(f"\nTimestamp: {datetime.now().isoformat()}")